# MM-Fit: RandomForest vs 1D-CNN (Klassifikation)

Dieses Notebook erstellt Sliding-Window-Segmente aus dem MM-Fit Datensatz und vergleicht eine RandomForest-Baseline
mit einem 1D-CNN.


## 1) Setup und Reproduzierbarkeit

 

In [ ]:
import os
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.ensemble import RandomForestClassifier

import tensorflow as tf
from tensorflow.keras import layers, models

# Reproduzierbarkeit
np.random.seed(42)
tf.random.set_seed(42)

# Pfade und Konfiguration
ROOT = "mm-fit"
TARGETS = ["pushups", "squats", "situps"]
label_map = {"pushups": 0, "squats": 1, "situps": 2}
inv_label_map = {v: k for k, v in label_map.items()}

SENSOR = "sw_r"
WIN = 128
STEP = 64

print("OK - Setup loaded")


OK - Setup loaded


/Users/kacharino/myProjects/Bachelor/.venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


**Datensatz**
- Root: `mm-fit/` mit Sessions `w00`–`w20`
- Pro Session: `*_sw_r_acc.npy`, `*_sw_r_gyr.npy`, `*_labels.csv`
- Labels: `start, end, reps, exercise` (nur pushups/squats/situps)


## 2) Sessions und Train/Test Split (w00–w15 vs w16–w20)


Ausblick: Train/Test Split variieren lassen mit einem Zufallswert.


In [ ]:
def list_sessions(root=ROOT):
    return sorted([
        d for d in os.listdir(root)
        if d.startswith("w") and os.path.isdir(os.path.join(root, d))
    ])

sessions = list_sessions(ROOT)
train_sessions = [s for s in sessions if int(s[1:]) <= 15]
test_sessions = [s for s in sessions if int(s[1:]) >= 16]

print("Anzahl Sessions:", len(sessions))
print("Train sessions:", train_sessions)
print("Test sessions:", test_sessions)


Anzahl Sessions: 21
Train sessions: ['w00', 'w01', 'w02', 'w03', 'w04', 'w05', 'w06', 'w07', 'w08', 'w09', 'w10', 'w11', 'w12', 'w13', 'w14', 'w15']
Test sessions: ['w16', 'w17', 'w18', 'w19', 'w20']


## 3) Laden einer Session (ACC + GYR, x/y/z)


In [ ]:
def load_one_session(wdir, sensor=SENSOR):
    base = os.path.join(ROOT, wdir)

    acc = np.load(os.path.join(base, f"{wdir}_{sensor}_acc.npy"))
    gyr = np.load(os.path.join(base, f"{wdir}_{sensor}_gyr.npy"))

    # x, y, z (Spalten 1 bis 3)
    acc_xyz = acc[:, 1:4].astype(np.float32)
    gyr_xyz = gyr[:, 1:4].astype(np.float32)

    labels = pd.read_csv(
        os.path.join(base, f"{wdir}_labels.csv"),
        header=None,
        names=["start", "end", "reps", "exercise"]
    )

    labels = labels[labels["exercise"].isin(TARGETS)].reset_index(drop=True)
    return acc_xyz, gyr_xyz, labels


## 4) Windowing (win=128, step=64)


In [ ]:
def make_windows(acc_xyz, gyr_xyz, labels_df, win=WIN, step=STEP):
    X, y = [], []
    for _, row in labels_df.iterrows():
        start, end = int(row["start"]), int(row["end"])
        lab = label_map[row["exercise"]]

        for i in range(start, end - win + 1, step):
            w_acc = acc_xyz[i:i+win]
            w_gyr = gyr_xyz[i:i+win]
            X.append(np.hstack([w_acc, w_gyr]))  # (win, 6)
            y.append(lab)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)


## 5) Dataset bauen

In [ ]:
def build_from_session(wdir):
    acc_xyz, gyr_xyz, labels = load_one_session(wdir)
    X, y = make_windows(acc_xyz, gyr_xyz, labels)
    return X, y


def build_dataset(session_list):
    X_all, y_all = [], []
    for wdir in session_list:
        X, y = build_from_session(wdir)
        if len(X) == 0:
            print("Skip (no windows):", wdir)
            continue
        X_all.append(X)
        y_all.append(y)
        print(wdir, "->", X.shape, np.unique(y, return_counts=True))
    return np.concatenate(X_all), np.concatenate(y_all)

X_train, y_train = build_dataset(train_sessions)
X_test, y_test = build_dataset(test_sessions)

print("\nFINAL:")
print("X_train:", X_train.shape, "y_train:", y_train.shape, np.unique(y_train, return_counts=True))
print("X_test :", X_test.shape,  "y_test :", y_test.shape,  np.unique(y_test, return_counts=True))


w00 -> (54, 128, 6) (array([0, 1, 2]), array([14, 18, 22]))
w01 -> (59, 128, 6) (array([0, 1, 2]), array([16, 20, 23]))
w02 -> (80, 128, 6) (array([0, 1, 2]), array([17, 27, 36]))
w03 -> (59, 128, 6) (array([0, 1, 2]), array([18, 18, 23]))
w04 -> (86, 128, 6) (array([0, 1, 2]), array([17, 31, 38]))
w05 -> (57, 128, 6) (array([0, 1, 2]), array([16, 22, 19]))
w06 -> (65, 128, 6) (array([0, 1, 2]), array([20, 22, 23]))
w07 -> (90, 128, 6) (array([0, 1, 2]), array([20, 29, 41]))
w08 -> (59, 128, 6) (array([0, 1, 2]), array([16, 21, 22]))
w09 -> (86, 128, 6) (array([0, 1, 2]), array([21, 28, 37]))
w10 -> (67, 128, 6) (array([0, 1, 2]), array([19, 23, 25]))
w11 -> (92, 128, 6) (array([0, 1, 2]), array([21, 29, 42]))
w12 -> (59, 128, 6) (array([0, 1, 2]), array([13, 21, 25]))
w13 -> (62, 128, 6) (array([0, 1, 2]), array([13, 20, 29]))
w14 -> (58, 128, 6) (array([0, 1, 2]), array([17, 19, 22]))
w15 -> (87, 128, 6) (array([0, 1, 2]), array([17, 29, 41]))
w16 -> (88, 128, 6) (array([0, 1, 2]), a

## 6) Normalisierung (train-basiert)


In [ ]:
mu = X_train.mean(axis=(0, 1), keepdims=True)
sigma = X_train.std(axis=(0, 1), keepdims=True) + 1e-8

X_train_n = (X_train - mu) / sigma
X_test_n  = (X_test  - mu) / sigma

print("Normalized shapes:", X_train_n.shape, X_test_n.shape)
print("Train mean ~", X_train_n.mean(), "Train std ~", X_train_n.std())


Normalized shapes: (1120, 128, 6) (443, 128, 6)
Train mean ~ 0.16523351 Train std ~ 0.98623985


In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)

class_weights = dict(zip(classes, weights))
print("class_weights:", class_weights)


class_weights: {np.int64(0): np.float64(1.3575757575757577), np.int64(1): np.float64(0.9902740937223696), np.int64(2): np.float64(0.7977207977207977)}


## 7) RandomForest 


In [ ]:
X_train_rf = X_train_n.reshape(X_train_n.shape[0], -1)
X_test_rf  = X_test_n.reshape(X_test_n.shape[0], -1)

rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train_rf, y_train)

y_pred_rf = rf.predict(X_test_rf)

print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_rf))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_rf,
    target_names=[inv_label_map[i] for i in sorted(inv_label_map)]
))


Random Forest Accuracy: 0.38826185101580135

Confusion Matrix:
[[  9   5 113]
 [ 22  16  99]
 [ 25   7 147]]

Classification Report:
              precision    recall  f1-score   support

     pushups       0.16      0.07      0.10       127
      squats       0.57      0.12      0.19       137
      situps       0.41      0.82      0.55       179

    accuracy                           0.39       443
   macro avg       0.38      0.34      0.28       443
weighted avg       0.39      0.39      0.31       443



## 8) 1D-CNN (Conv1D + BatchNorm + Dropout)


In [ ]:
def build_cnn(input_shape=(128, 6), num_classes=3):
    inputs = layers.Input(shape=input_shape)

    x = layers.Conv1D(64, 7, padding="same")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, 5, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(256, 3, padding="same")(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)

    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

cnn = build_cnn()
cnn.summary()


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 6)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 128, 64)        │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 64, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 64, 128)        │        41,088 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 128)        │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 32, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 32, 256)        │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 32, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 177,475 (693.26 KB)

 Trainable params: 176,579 (689.76 KB)

 Non-trainable params: 896 (3.50 KB)

**Hinweis (Limitierung):** Der Train/Val-Split erfolgt auf Window-Ebene innerhalb der Trainings-Sessions. Das kann zu sehr ähnlichen Fenstern in Train und Val führen und damit die Validierung leicht optimistisch verzerren.


In [ ]:
# Data augmentation: jitter + scaling
# (applied on-the-fly to training windows)
def augment(x, y):
    noise = tf.random.normal(tf.shape(x), mean=0.0, stddev=0.02)
    scale = tf.random.uniform([tf.shape(x)[0], 1, 1], 0.9, 1.1)
    x = x * scale + noise
    return x, y

batch_size = 32

# Train/val split on window level (shuffled, reproducible)
rng = np.random.default_rng(42)
indices = rng.permutation(len(X_train_n))
val_size = int(0.2 * len(indices))
val_idx = indices[:val_size]
train_idx = indices[val_size:]

X_tr, y_tr = X_train_n[train_idx], y_train[train_idx]
X_val, y_val = X_train_n[val_idx], y_train[val_idx]

train_ds = tf.data.Dataset.from_tensor_slices((X_tr, y_tr))
train_ds = train_ds.shuffle(8192).batch(batch_size).map(augment).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy",
        patience=10,
        restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=4,
        min_lr=1e-5
    )
]

history = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=80,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1
)


Epoch 1/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 2s 13ms/step - accuracy: 0.3828 - loss: 1.2553 - val_accuracy: 0.4286 - val_loss: 1.0471 - learning_rate: 0.0010
Epoch 2/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4420 - loss: 1.1189 - val_accuracy: 0.5089 - val_loss: 1.0145 - learning_rate: 0.0010
Epoch 3/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4810 - loss: 1.0051 - val_accuracy: 0.5580 - val_loss: 0.9463 - learning_rate: 0.0010
Epoch 4/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5011 - loss: 0.9902 - val_accuracy: 0.5714 - val_loss: 0.9043 - learning_rate: 0.0010
Epoch 5/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5011 - loss: 0.9525 - val_accuracy: 0.5938 - val_loss: 0.8832 - learning_rate: 0.0010
Epoch 6/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5112 - loss: 0.9607 - val_accuracy: 0.5848 - val_loss: 0.8524 - learning_rate: 0.0010
Epoch 7/80
28/28 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5257 - loss: 0.9379 - val_accuracy:

## 8) Evaluation (Accuracy, Confusion Matrix)



In [ ]:
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    _HAS_SEABORN = True
except Exception:
    _HAS_SEABORN = False

# CNN Evaluation
probs = cnn.predict(X_test_n, verbose=0)
y_pred_cnn = probs.argmax(axis=1)

print("CNN Accuracy:", accuracy_score(y_test, y_pred_cnn))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_cnn))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred_cnn,
    target_names=[inv_label_map[i] for i in sorted(inv_label_map)],
    zero_division=0
))


CNN Accuracy: 0.4040632054176072

Confusion Matrix:
[[  0   0 127]
 [  0   0 137]
 [  0   0 179]]

Classification Report:
              precision    recall  f1-score   support

     pushups       0.00      0.00      0.00       127
      squats       0.00      0.00      0.00       137
      situps       0.40      1.00      0.58       179

    accuracy                           0.40       443
   macro avg       0.13      0.33      0.19       443
weighted avg       0.16      0.40      0.23       443

